# Lustre Deployment on AWS (Configurable Topology)

## Prerequisites

- Python packages: boto3, paramiko, requests
- AWS credentials configured
- SSH key pair in AWS and locally
- Security group allowing SSH and Lustre traffic
- Sufficient AWS quotas for instances, volumes, and EIPs

**CRITICAL: All resources must be in the same Availability Zone**

## Configurable Parameters

This notebook supports configurable numbers of MDS, OSS, and client instances,
matching the EXAScaler Cloud CloudFormation template parameters. AMIs and Maven
are resolved dynamically at runtime.

In [ ]:
# Version Information
from __future__ import annotations

notebook_version = '6.0'
last_updated = '2026-03-17'
lustre_version = '2.15.8'

print(
    f'Notebook v{notebook_version} | Lustre {lustre_version} | {last_updated}',
)

## Prerequisites Setup

Install Python packages, import shared helpers, and verify AWS credentials via STS.

In [ ]:
# Package Installation
import subprocess
import sys

sys.path.insert(0, '.')

for pkg in ['boto3', 'paramiko', 'requests']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        )

print('✓ All required packages are available')

In [ ]:
# Import Required Packages
import subprocess
import sys
from datetime import datetime

import boto3
import paramiko
from botocore.exceptions import ClientError
from botocore.exceptions import NoCredentialsError
from deploy_common import install_filebench
from deploy_common import install_icicle
from deploy_common import launch_instances_parallel
from deploy_common import load_cluster_state
from deploy_common import load_deploy_config
from deploy_common import read_github_token
from deploy_common import reboot_and_wait
from deploy_common import resolve_maven_version
from deploy_common import resolve_rhel_ami
from deploy_common import resolve_ubuntu_ami
from deploy_common import run_parallel
from deploy_common import save_cluster_state
from deploy_common import setup_logger
from deploy_common import setup_permissions
from deploy_common import setup_rhel
from deploy_common import setup_ubuntu
from deploy_common import ssh_exec
from deploy_common import update_ssh_config
from deploy_common import validate_aws_config
from deploy_common import wait_for_ssh

b3 = boto3.__version__
pm = paramiko.__version__
py = sys.version.split()[0]
print(f'boto3={b3} paramiko={pm} python={py}')
print('✓ All packages imported')

In [ ]:
# AWS Credential Verification
try:
    sts = boto3.client('sts', region_name='us-east-1')
    identity = sts.get_caller_identity()
    print(f'✓ Account: {identity["Account"]}')
    print(f'  ARN:     {identity["Arn"]}')
except (NoCredentialsError, ClientError) as e:
    raise RuntimeError(
        f'AWS credentials not configured or invalid: {e}',
    ) from e

## Logger Setup

Configure a file-based logger for the deployment session.

In [ ]:
# Configure File-Based Logger
import logging

LOG_LEVEL = logging.INFO  # Change to logging.DEBUG for verbose output

logger, log_file = setup_logger('deploy-lustre', LOG_LEVEL)

## Configuration

Parameters are loaded from `deploy-config.yaml` via `load_deploy_config('lustre')`.

In [ ]:
# Configuration Variables
import random
import string

config = load_deploy_config('lustre')

# --- Cluster ID: load from state file or generate new ---
_state = load_cluster_state('lustre-state.json', logger=logger)
if _state:
    CLUSTER_ID = _state['cluster_id']
    print(f'Resuming existing cluster: {CLUSTER_ID}')
else:
    CLUSTER_ID = ''.join(random.choices(string.ascii_lowercase, k=5))
    _state = {
        'cluster_id': CLUSTER_ID,
        'created_at': datetime.now().isoformat(),
    }
    save_cluster_state(_state, 'lustre-state.json', logger=logger)
    print(f'New cluster created: {CLUSTER_ID}')

# --- Derived values ---
NUM_OSS = config['num_oss']
NUM_MDS = config['num_mds']
NUM_CLIENTS = config['num_compute_clients']
NUM_OST_PER_OSS = config['num_ost_per_server']
OST_VOLUME_SIZE_GB = max(
    10,
    config['fs_capacity_gb'] // (NUM_OSS * NUM_OST_PER_OSS),
)

# Convenience aliases
FSNAME = config['fsname']
AWS_REGION = config['aws_region']
AWS_AZ = config['aws_az']
KEY_NAME = config['ssh_key_name']
KEY_PATH = config['ssh_key_path']
# The security group implicitly determines the VPC — no vpc_id needed.
SECURITY_GROUP = config['security_group']
PLACEMENT_GROUP = config['placement_group']
SERVER_SSH_USER = config['server_ssh_user']
CLIENT_SSH_USER = config['client_ssh_user']

logger.info('Configuration loaded')
print(f'Cluster ID: {CLUSTER_ID}')
total_osts = NUM_OSS * NUM_OST_PER_OSS
print(
    f'Topology: 1 MGS + {NUM_MDS} MDS + {NUM_OSS} OSS'
    f' ({total_osts} OSTs) + {NUM_CLIENTS} clients',
)
print(f'OST volume size (derived): {OST_VOLUME_SIZE_GB} GB each')

In [ ]:
# Configuration Validation
MAX_FSNAME_LEN = 8
AWS_EIP_DEFAULT_LIMIT = 5

validate_aws_config(KEY_PATH, SECURITY_GROUP, AWS_REGION, AWS_AZ)

assert FSNAME.isalnum(), f"fsname must be alphanumeric, got: '{FSNAME}'"
assert 1 <= len(FSNAME) <= MAX_FSNAME_LEN, (
    f"fsname must be 1-{MAX_FSNAME_LEN} chars, got: '{FSNAME}'"
)
assert NUM_OSS >= 1, f'num_oss must be >= 1, got: {NUM_OSS}'
assert NUM_MDS >= 1, f'num_mds must be >= 1, got: {NUM_MDS}'
assert NUM_OST_PER_OSS >= 1, (
    f'num_ost_per_server must be >= 1, got: {NUM_OST_PER_OSS}'
)

total_eips = 1 + (NUM_MDS - 1) + NUM_OSS + NUM_CLIENTS
if total_eips > AWS_EIP_DEFAULT_LIMIT:
    print(
        f'⚠ {total_eips} EIPs needed'
        f' (default AWS limit is {AWS_EIP_DEFAULT_LIMIT})',
    )

print(f'✓ Configuration validated ({total_eips} EIPs needed)')

## Dynamic AMI and Maven Resolution

Query AWS for the latest RHEL 8 and Ubuntu 24.04 AMIs in the target region,
and resolve the latest Maven version from Maven Central.

In [ ]:
# Dynamic AMI Lookup
ec2 = boto3.client('ec2', region_name=AWS_REGION)

AMI_ID, rhel_name = resolve_rhel_ami(ec2, AWS_REGION)
print(f'✓ RHEL AMI: {rhel_name}')
print(f'  ID: {AMI_ID}')

UBUNTU_AMI_ID, ubuntu_name = resolve_ubuntu_ami(ec2, AWS_REGION)
print(f'\n✓ Ubuntu AMI: {ubuntu_name}')
print(f'  ID: {UBUNTU_AMI_ID}')

logger.info(f'Resolved AMIs: RHEL={AMI_ID}, Ubuntu={UBUNTU_AMI_ID}')

In [ ]:
# Dynamic Maven Version Resolution
MAVEN_VERSION, MAVEN_URL = resolve_maven_version()
print(f'✓ Maven version: {MAVEN_VERSION}')
print(f'  URL: {MAVEN_URL}')
logger.info(f'Resolved Maven: {MAVEN_VERSION}')

## Deployment Steps

### Step 1: Launch All Instances in Parallel

Launch MGS+MDS, additional MDS, OSS (each with data volumes), and client
instances via `launch_instances_parallel`. Allocates EIPs and updates SSH config.

In [ ]:
# Step 1: Launch ALL instances (servers + clients) in parallel
start_time = datetime.now()

MGS_MDS_INSTANCE_NAME = f'{FSNAME}-{CLUSTER_ID}-MGS+MDS0'

# Build launch specs for all nodes
all_launch_specs = []

# MGS+MDS0
all_launch_specs.append(
    {
        'name': MGS_MDS_INSTANCE_NAME,
        'ami_id': AMI_ID,
        'instance_type': config['mgs_instance_type'],
        'data_volumes': [('/dev/sdf', config['mds_volume_size_gb'])],
        'placement_group': PLACEMENT_GROUP,
        'ssh_user': SERVER_SSH_USER,
    },
)
# Additional MDS nodes
for i in range(1, NUM_MDS):
    all_launch_specs.append(
        {
            'name': f'{FSNAME}-{CLUSTER_ID}-MDS{i}',
            'ami_id': AMI_ID,
            'instance_type': config['mds_instance_type'],
            'data_volumes': [('/dev/sdf', config['mds_volume_size_gb'])],
            'placement_group': PLACEMENT_GROUP,
            'ssh_user': SERVER_SSH_USER,
        },
    )
# OSS nodes
for i in range(NUM_OSS):
    all_launch_specs.append(
        {
            'name': f'{FSNAME}-{CLUSTER_ID}-OSS{i}',
            'ami_id': AMI_ID,
            'instance_type': config['oss_instance_type'],
            'data_volumes': [
                (f'/dev/sd{chr(ord("f") + j)}', OST_VOLUME_SIZE_GB)
                for j in range(NUM_OST_PER_OSS)
            ],
            'placement_group': PLACEMENT_GROUP,
            'ssh_user': SERVER_SSH_USER,
        },
    )
# Client nodes
for i in range(NUM_CLIENTS):
    all_launch_specs.append(
        {
            'name': f'{FSNAME}-{CLUSTER_ID}-client{i}',
            'ami_id': UBUNTU_AMI_ID,
            'instance_type': config['client_instance_type'],
            'data_volumes': [],
            'placement_group': '',
            'ssh_user': CLIENT_SSH_USER,
        },
    )

results = launch_instances_parallel(
    ec2,
    all_launch_specs,
    key_name=KEY_NAME,
    security_group=SECURITY_GROUP,
    availability_zone=AWS_AZ,
    root_volume_size=config['root_volume_size'],
    logger=logger,
)

# --- Extract results into typed node lists ---
mgs_mds_result = results[MGS_MDS_INSTANCE_NAME]
MGS_MDS_INSTANCE_ID = mgs_mds_result['instance_id']
MGS_MDS_EIP_ALLOCATION = mgs_mds_result['eip_allocation']
MGS_MDS_EIP = mgs_mds_result['eip']
MGS_MDS_PRIVATE_IP = mgs_mds_result['private_ip']
MGS_MDS_PRIVATE_DNS = mgs_mds_result['private_dns']
MGS_PRIVATE_IP = MGS_MDS_PRIVATE_IP

mds_nodes = []
for i in range(1, NUM_MDS):
    name = f'{FSNAME}-{CLUSTER_ID}-MDS{i}'
    mds_nodes.append(results[name])

oss_nodes = []
for i in range(NUM_OSS):
    name = f'{FSNAME}-{CLUSTER_ID}-OSS{i}'
    oss_nodes.append(results[name])

client_nodes = []
for i in range(NUM_CLIENTS):
    name = f'{FSNAME}-{CLUSTER_ID}-client{i}'
    client_nodes.append(results[name])

# --- Update SSH config for all nodes at once ---
ssh_instances = [
    {
        'name': MGS_MDS_INSTANCE_NAME,
        'ip': MGS_MDS_EIP,
        'user': SERVER_SSH_USER,
        'key_path': KEY_PATH,
    },
]
for node in mds_nodes:
    ssh_instances.append(
        {
            'name': node['name'],
            'ip': node['eip'],
            'user': SERVER_SSH_USER,
            'key_path': KEY_PATH,
        },
    )
for node in oss_nodes:
    ssh_instances.append(
        {
            'name': node['name'],
            'ip': node['eip'],
            'user': SERVER_SSH_USER,
            'key_path': KEY_PATH,
        },
    )
for node in client_nodes:
    ssh_instances.append(
        {
            'name': node['name'],
            'ip': node['eip'],
            'user': CLIENT_SSH_USER,
            'key_path': KEY_PATH,
        },
    )
update_ssh_config(ssh_instances, 'Lustre', logger=logger)

# --- Save state ---
_state['mgs_mds'] = {
    'instance_id': MGS_MDS_INSTANCE_ID,
    'name': MGS_MDS_INSTANCE_NAME,
    'eip': MGS_MDS_EIP,
    'private_ip': MGS_MDS_PRIVATE_IP,
    'eip_allocation': MGS_MDS_EIP_ALLOCATION,
}
_state['mds_nodes'] = [
    {
        'name': n['name'],
        'instance_id': n['instance_id'],
        'eip': n['eip'],
        'private_ip': n['private_ip'],
    }
    for n in mds_nodes
]
_state['oss_nodes'] = [
    {
        'name': n['name'],
        'instance_id': n['instance_id'],
        'eip': n['eip'],
        'private_ip': n['private_ip'],
    }
    for n in oss_nodes
]
_state['client_nodes'] = [
    {
        'name': n['name'],
        'instance_id': n['instance_id'],
        'eip': n['eip'],
        'private_ip': n.get('private_ip'),
    }
    for n in client_nodes
]
save_cluster_state(_state, 'lustre-state.json', logger=logger)

total = 1 + len(mds_nodes) + len(oss_nodes) + len(client_nodes)
logger.info(f'Step 1 completed: {total} instances launched in parallel')

### Step 2: Parallel Node Setup

Server pipeline (RHEL install, reboot, verify, Lustre format) runs in parallel
with client pipeline Phase A (Lustre client install, reboot, Ubuntu setup,
Firebench, Icicle). Client Lustre mount happens in Step 3 after servers are ready.

In [ ]:
# Step 2: Parallel server pipeline + client Phase A

# --- Shared client config ---
GITHUB_TOKEN = read_github_token()

GIT_USERNAME = config['git_username']
GIT_REPO = config['git_repo']
GIT_BRANCH = config['git_branch']

FSX_KEY_URL = (
    'https://fsx-lustre-client-repo-public-keys'
    '.s3.amazonaws.com/fsx-ubuntu-public-key.asc'
)
FSX_KEYRING = '/usr/share/keyrings/fsx-ubuntu-public-key.gpg'
FSX_REPO_URL = (
    'https://fsx-lustre-client-repo.s3.amazonaws.com/ubuntu jammy main'
)
FILEBENCH_VERSION = '1.5-alpha3'
FB_BASE = 'https://github.com/filebench/filebench/releases/download'
FILEBENCH_URL = (
    f'{FB_BASE}/{FILEBENCH_VERSION}/filebench-{FILEBENCH_VERSION}.tar.gz'
)


# --- Helper functions for Lustre server configuration ---


def configure_mgs_mdt(ssh, eip):
    """Format and mount combined MGS+MDT0."""
    mdt_dev = '/dev/nvme1n1'
    logger.info(f'Formatting {mdt_dev} as MGS+MDT0')

    ssh_exec(ssh, eip, f'sudo umount {mdt_dev} 2>/dev/null || true')
    ssh_exec(ssh, eip, f'sudo wipefs -a {mdt_dev} 2>/dev/null || true')
    ssh_exec(
        ssh,
        eip,
        f'sudo dd if=/dev/zero of={mdt_dev}'
        ' bs=1M count=10 conv=fsync 2>/dev/null && sudo sync',
    )

    exit_code, output, error = ssh_exec(
        ssh,
        eip,
        f'sudo mkfs.lustre --fsname={FSNAME} --mgs --mdt --index=0 {mdt_dev}',
        timeout=60,
    )
    if exit_code != 0:
        raise RuntimeError(f'Failed to format MGS+MDT: {error}')
    logger.info('✓ MGS+MDT0 formatted')
    print(output)

    ssh_exec(ssh, eip, 'sudo mkdir -p /mnt/mgs_mdt')
    exit_code, _, error = ssh_exec(
        ssh,
        eip,
        f'sudo mount -t lustre {mdt_dev} /mnt/mgs_mdt',
        timeout=90,
    )
    if exit_code != 0:
        raise RuntimeError(f'Failed to mount MDT: {error}')
    logger.info('✓ MGS+MDT0 mounted')

    exit_code, fstab_check, _ = ssh_exec(
        ssh,
        eip,
        f"grep -E '{mdt_dev}.*mgs_mdt' /etc/fstab || echo 'NOT_FOUND'",
    )
    if 'NOT_FOUND' in fstab_check:
        fstab_entry = f'{mdt_dev} /mnt/mgs_mdt lustre defaults,_netdev 0 0'
        ssh_exec(
            ssh,
            eip,
            f"echo '{fstab_entry}' | sudo tee -a /etc/fstab",
        )
    ssh_exec(ssh, eip, 'sudo lctl dl')


def configure_mds(i, node, mgs_private_ip):
    """Format and mount standalone MDT."""
    ssh, eip = node['ssh'], node['eip']
    mdt_dev = '/dev/nvme1n1'
    logger.info(f'Configuring MDT{i} on {node["name"]}')

    ssh_exec(ssh, eip, f'sudo umount {mdt_dev} 2>/dev/null || true')
    ssh_exec(ssh, eip, f'sudo wipefs -a {mdt_dev} 2>/dev/null || true')
    ssh_exec(
        ssh,
        eip,
        f'sudo dd if=/dev/zero of={mdt_dev}'
        ' bs=1M count=10 conv=fsync 2>/dev/null',
    )
    ssh_exec(ssh, eip, 'sudo sync')

    exit_code, _, error = ssh_exec(
        ssh,
        eip,
        f'sudo mkfs.lustre --fsname={FSNAME} --mdt --index={i} '
        f'--mgsnode={mgs_private_ip}@tcp {mdt_dev}',
        timeout=60,
    )
    if exit_code != 0:
        raise RuntimeError(
            f'Failed to format MDT{i} on {node["name"]}: {error}',
        )
    logger.info(f'✓ MDT{i} formatted')

    mount_point = f'/mnt/mdt{i}'
    ssh_exec(ssh, eip, f'sudo mkdir -p {mount_point}')
    exit_code, _, error = ssh_exec(
        ssh,
        eip,
        f'sudo mount -t lustre {mdt_dev} {mount_point}',
        timeout=90,
    )
    if exit_code != 0:
        raise RuntimeError(
            f'Failed to mount MDT{i} on {node["name"]}: {error}',
        )
    logger.info(f'✓ MDT{i} mounted at {mount_point}')

    exit_code, fstab_check, _ = ssh_exec(
        ssh,
        eip,
        f"grep '{mdt_dev}' /etc/fstab || echo 'NOT_FOUND'",
    )
    if 'NOT_FOUND' in fstab_check:
        fstab_entry = f'{mdt_dev} {mount_point} lustre defaults,_netdev 0 0'
        ssh_exec(
            ssh,
            eip,
            f"echo '{fstab_entry}' | sudo tee -a /etc/fstab",
        )
    ssh_exec(ssh, eip, 'sudo lctl dl')


def configure_oss(node, start_ost_index, mgs_private_ip):
    """Format and mount all OSTs on an OSS node."""
    ssh, eip = node['ssh'], node['eip']

    for j in range(NUM_OST_PER_OSS):
        idx = start_ost_index + j
        dev = f'/dev/nvme{j + 1}n1'
        mount_point = f'/mnt/ost{idx}'
        logger.info(f'Configuring OST{idx} on {node["name"]}:{dev}')

        ssh_exec(ssh, eip, f'sudo umount {dev} 2>/dev/null || true')
        ssh_exec(ssh, eip, f'sudo wipefs -a {dev} 2>/dev/null || true')
        ssh_exec(
            ssh,
            eip,
            f'sudo dd if=/dev/zero of={dev}'
            ' bs=1M count=10 conv=fsync 2>/dev/null',
        )
        ssh_exec(ssh, eip, 'sudo sync')

        exit_code, _, error = ssh_exec(
            ssh,
            eip,
            f'sudo mkfs.lustre --fsname={FSNAME}'
            f' --ost --index={idx}'
            f' --mgsnode={mgs_private_ip}@tcp {dev}',
            timeout=60,
        )
        if exit_code != 0:
            raise RuntimeError(
                f'Failed to format OST{idx} on {node["name"]}: {error}',
            )
        logger.info(f'✓ OST{idx} formatted')

        ssh_exec(ssh, eip, f'sudo mkdir -p {mount_point}')
        exit_code, _, error = ssh_exec(
            ssh,
            eip,
            f'sudo mount -t lustre {dev} {mount_point}',
            timeout=90,
        )
        if exit_code != 0:
            raise RuntimeError(
                f'Failed to mount OST{idx} on {node["name"]}: {error}',
            )
        logger.info(f'✓ OST{idx} mounted at {mount_point}')

        exit_code, fstab_check, _ = ssh_exec(
            ssh,
            eip,
            f"grep '{dev}' /etc/fstab || echo 'NOT_FOUND'",
        )
        if 'NOT_FOUND' in fstab_check:
            fstab_entry = f'{dev} {mount_point} lustre defaults,_netdev 0 0'
            ssh_exec(
                ssh,
                eip,
                f"echo '{fstab_entry}' | sudo tee -a /etc/fstab",
            )

    ssh_exec(ssh, eip, 'df -h | grep ost')
    ssh_exec(ssh, eip, 'sudo lctl dl')


# --- Server pipeline sub-steps ---


def _server_rhel_setup():
    """RHEL setup on all servers (parallel). Returns mgs_ssh."""

    def setup_server_node(eip, name):
        ssh = wait_for_ssh(eip, KEY_PATH, SERVER_SSH_USER)
        if not ssh:
            raise Exception(f'SSH timeout for {name} at {eip}')
        ssh_exec(ssh, eip, f'sudo hostnamectl set-hostname {name}')
        ssh_exec(ssh, eip, 'hostname')
        logger.info(f'Running RHEL setup on {name}')
        if not setup_rhel(ssh, eip, logger=logger):
            raise Exception(f'RHEL setup failed for {name}')
        return ssh

    tasks = [
        (
            MGS_MDS_INSTANCE_NAME,
            lambda: setup_server_node(MGS_MDS_EIP, MGS_MDS_INSTANCE_NAME),
        ),
    ]
    for node in mds_nodes:
        tasks.append(
            (
                node['name'],
                lambda n=node: setup_server_node(n['eip'], n['name']),
            ),
        )
    for node in oss_nodes:
        tasks.append(
            (
                node['name'],
                lambda n=node: setup_server_node(n['eip'], n['name']),
            ),
        )
    results = run_parallel(tasks, logger=logger)

    mgs_ssh = results[MGS_MDS_INSTANCE_NAME]
    for node in mds_nodes:
        node['ssh'] = results[node['name']]
    for node in oss_nodes:
        node['ssh'] = results[node['name']]
    logger.info('RHEL setup completed (parallel)')
    return mgs_ssh


def _server_reboot(mgs_ssh):
    """Reboot all servers (parallel). Returns new mgs_ssh."""

    def reboot_node(node_label, ssh, eip):
        new_ssh = reboot_and_wait(
            ssh,
            eip,
            KEY_PATH,
            SERVER_SSH_USER,
            logger=logger,
        )
        if not new_ssh:
            raise Exception(
                f'Failed to reconnect to {node_label} after reboot',
            )
        return new_ssh

    all_for_reboot = [(MGS_MDS_INSTANCE_NAME, mgs_ssh, MGS_MDS_EIP)]
    for node in mds_nodes:
        all_for_reboot.append((node['name'], node['ssh'], node['eip']))
    for node in oss_nodes:
        all_for_reboot.append((node['name'], node['ssh'], node['eip']))

    tasks = [
        (label, lambda lb=label, s=ssh, e=eip: reboot_node(lb, s, e))
        for label, ssh, eip in all_for_reboot
    ]
    results = run_parallel(tasks, logger=logger)

    mgs_ssh = results[MGS_MDS_INSTANCE_NAME]
    for node in mds_nodes:
        node['ssh'] = results[node['name']]
    for node in oss_nodes:
        node['ssh'] = results[node['name']]
    logger.info('Reboot completed (parallel)')
    return mgs_ssh


def _server_verify(mgs_ssh):
    """Post-reboot verification on all servers (parallel)."""

    def verify_node(node_label, ssh, eip):
        logger.info(f'Loading kernel modules on {node_label}')
        ssh_exec(ssh, eip, 'sudo modprobe lnet', timeout=30, logger=logger)
        ssh_exec(
            ssh,
            eip,
            'sudo modprobe -v lustre',
            timeout=30,
            logger=logger,
        )
        _, output, _ = ssh_exec(
            ssh,
            eip,
            'sudo lctl get_param version',
            timeout=15,
            logger=logger,
        )
        if output:
            logger.info(f'{node_label} Lustre version: {output.strip()}')

    all_server_nodes = [('MGS+MDS0', mgs_ssh, MGS_MDS_EIP)]
    for node in mds_nodes:
        all_server_nodes.append((node['name'], node['ssh'], node['eip']))
    for node in oss_nodes:
        all_server_nodes.append((node['name'], node['ssh'], node['eip']))

    tasks = [
        (label, lambda lb=label, s=ssh, e=eip: verify_node(lb, s, e))
        for label, ssh, eip in all_server_nodes
    ]
    run_parallel(tasks, logger=logger)
    logger.info('Post-reboot verification completed (parallel)')


def _server_format_lustre(mgs_ssh):
    """Format MGS+MDT0, then additional MDS + OSS in parallel."""
    configure_mgs_mdt(mgs_ssh, MGS_MDS_EIP)

    format_tasks = []
    for i, node in enumerate(mds_nodes, start=1):
        format_tasks.append(
            (
                f'MDT{i}',
                lambda i=i, n=node: configure_mds(i, n, MGS_PRIVATE_IP),
            ),
        )
    ost_start = 0
    for node in oss_nodes:
        format_tasks.append(
            (
                node['name'],
                lambda n=node, s=ost_start: configure_oss(
                    n,
                    s,
                    MGS_PRIVATE_IP,
                ),
            ),
        )
        ost_start += NUM_OST_PER_OSS

    if format_tasks:
        run_parallel(format_tasks, logger=logger)


def server_pipeline():
    """Full server setup: RHEL → reboot → verify → MGS format → MDS/OSS."""
    mgs_ssh = _server_rhel_setup()
    mgs_ssh = _server_reboot(mgs_ssh)
    _server_verify(mgs_ssh)
    _server_format_lustre(mgs_ssh)
    logger.info('Server pipeline completed')
    return mgs_ssh


# --- Client Phase A (MGS-independent setup) ---


def client_phase_a(node):
    """Lustre client install, reboot, Ubuntu setup, Filebench, Icicle."""
    eip = node['eip']
    name = node['name']

    ssh = wait_for_ssh(eip, KEY_PATH, CLIENT_SSH_USER)
    if not ssh:
        raise Exception(f'SSH timeout for {name} at {eip}')
    ssh_exec(ssh, eip, f'sudo hostnamectl set-hostname {name}')

    # Lustre client install
    logger.info(f'Installing Lustre client on {name}')
    ssh_exec(ssh, eip, 'sudo apt-get update', timeout=60)
    ssh_exec(
        ssh,
        eip,
        f'wget -O - {FSX_KEY_URL}'
        f' | gpg --dearmor | sudo tee {FSX_KEYRING} >/dev/null',
    )
    ssh_exec(
        ssh,
        eip,
        "sudo bash -c '"
        f'echo "deb [signed-by={FSX_KEYRING}] '
        f'{FSX_REPO_URL}" > '
        '/etc/apt/sources.list.d/fsxlustreclientrepo.list'
        " && apt-get update'",
        timeout=60,
    )
    ssh_exec(
        ssh,
        eip,
        'sudo apt install -y linux-aws lustre-client-modules-aws',
        timeout=300,
    )

    # Reboot
    logger.info(f'Rebooting {name}')
    ssh = reboot_and_wait(ssh, eip, KEY_PATH, CLIENT_SSH_USER, logger=logger)
    if not ssh:
        raise Exception(f'Failed to reconnect to {name} after reboot')

    ssh_exec(
        ssh,
        eip,
        'sudo apt install -y lustre-client-modules-aws',
        timeout=150,
    )
    ssh_exec(ssh, eip, 'sudo modprobe lnet && sudo modprobe lustre')

    # Ubuntu setup (Maven, packages, AWS CLI, credentials)
    logger.info(
        f'Running Ubuntu setup on {name} (Maven {MAVEN_VERSION})',
    )
    if not setup_ubuntu(
        ssh,
        eip,
        MAVEN_VERSION,
        MAVEN_URL,
        CLIENT_SSH_USER,
        AWS_REGION,
        logger=logger,
    ):
        raise RuntimeError(f'Ubuntu setup failed on {name}')

    # Filebench
    logger.info(f'Installing Filebench on {name}')
    if not install_filebench(
        ssh,
        eip,
        FILEBENCH_VERSION,
        FILEBENCH_URL,
        logger=logger,
    ):
        raise RuntimeError(f'Filebench install failed on {name}')
    logger.info(f'✓ Filebench installed on {name}')

    # Icicle
    logger.info(f'Installing Icicle on {name}')
    if not install_icicle(
        ssh,
        eip,
        GITHUB_TOKEN,
        GIT_USERNAME,
        GIT_REPO,
        GIT_BRANCH,
        python_version='3.11',
        logger=logger,
    ):
        raise Exception(f'Icicle installation failed on {name}')
    logger.info(f'✓ Icicle installed on {name}')

    node['ssh'] = ssh
    return ssh


# --- Run server pipeline + all client Phase A in parallel ---
tasks = [('servers', server_pipeline)]
for node in client_nodes:
    tasks.append(
        (node['name'], lambda n=node: client_phase_a(n)),
    )

results = run_parallel(tasks, logger=logger)

mgs_ssh = results['servers']

logger.info('Step 2 completed: server + client Phase A (parallel)')

### Step 3: Mount Clients, Changelog, and Verification

Mount Lustre on all clients, create per-MDT striped subdirectories, enable
changelog on every MDT, then verify with `lfs df`, `lfs check`, and file I/O tests.

In [ ]:
# Step 3: Mount Lustre on clients + changelog on all MDTs + verification

# --- Mount Lustre on all clients (needs MGS, fast) ---
for node in client_nodes:
    ssh, eip, name = node['ssh'], node['eip'], node['name']
    ssh_exec(ssh, eip, f'sudo mkdir -p /mnt/{FSNAME}')
    logger.info(f'Mounting Lustre on {name}')
    exit_code, _, error = ssh_exec(
        ssh,
        eip,
        f'sudo mount -t lustre {MGS_PRIVATE_IP}@tcp:/{FSNAME} /mnt/{FSNAME}',
        timeout=90,
    )
    if exit_code != 0:
        raise RuntimeError(f'Failed to mount Lustre on {name}: {error}')
    logger.info(f'✓ Lustre mounted on {name}')

    setup_permissions(
        ssh,
        eip,
        f'/mnt/{FSNAME}',
        CLIENT_SSH_USER,
        logger=logger,
    )

    # Create per-MDT subdirectories striped to their respective MDTs.
    for mdt_idx in range(NUM_MDS):
        mdt_dir = f'/mnt/{FSNAME}/mdt{mdt_idx}'
        ssh_exec(
            ssh,
            eip,
            f'sudo lfs mkdir -i {mdt_idx} {mdt_dir}',
            logger=logger,
        )
        setup_permissions(
            ssh,
            eip,
            mdt_dir,
            CLIENT_SSH_USER,
            logger=logger,
        )
    logger.info(f'✓ {NUM_MDS} MDT subdirectories created on {name}')

# --- Changelog on ALL MDTs ---
logger.info('Enabling changelog on all MDTs')

all_mdt_nodes = [(0, mgs_ssh, MGS_MDS_EIP)]
for i, node in enumerate(mds_nodes, start=1):
    all_mdt_nodes.append((i, node['ssh'], node['eip']))

for mdt_idx, ssh, eip in all_mdt_nodes:
    mdt_name = f'{FSNAME}-MDT{mdt_idx:04d}'
    logger.info(f'Registering changelog on {mdt_name}')
    ssh_exec(
        ssh,
        eip,
        f'sudo lctl --device {mdt_name} changelog_register',
        logger=logger,
    )
    ssh_exec(
        ssh,
        eip,
        f'sudo lctl set_param mdd.{mdt_name}.changelog_mask=ALL',
        logger=logger,
    )
    ssh_exec(
        ssh,
        eip,
        f'sudo lctl get_param mdd.{mdt_name}.changelog_mask',
        logger=logger,
    )
    ssh_exec(
        ssh,
        eip,
        f'sudo lctl get_param mdd.{mdt_name}.changelog_users',
        logger=logger,
    )

logger.info(f'Changelog enabled on {len(all_mdt_nodes)} MDTs')

# --- Cross-node connectivity verification ---
logger.info('Testing connectivity between nodes')

for node in oss_nodes:
    exit_code, _, _ = ssh_exec(
        mgs_ssh,
        MGS_MDS_EIP,
        f'ping -c 3 {node["eip"]}',
        logger=logger,
    )
    if exit_code == 0:
        logger.info(f'✓ MGS+MDS0 can reach {node["name"]}')
    exit_code, _, _ = ssh_exec(
        node['ssh'],
        node['eip'],
        f'ping -c 3 {MGS_MDS_EIP}',
        logger=logger,
    )
    if exit_code == 0:
        logger.info(f'✓ {node["name"]} can reach MGS+MDS0')

for node in mds_nodes:
    exit_code, _, _ = ssh_exec(
        mgs_ssh,
        MGS_MDS_EIP,
        f'ping -c 3 {node["eip"]}',
        logger=logger,
    )
    if exit_code == 0:
        logger.info(f'✓ MGS+MDS0 can reach {node["name"]}')

# Lustre health on all servers
all_server_nodes = [('MGS+MDS0', mgs_ssh, MGS_MDS_EIP)]
for node in mds_nodes:
    all_server_nodes.append((node['name'], node['ssh'], node['eip']))
for node in oss_nodes:
    all_server_nodes.append((node['name'], node['ssh'], node['eip']))

for label, ssh, eip in all_server_nodes:
    logger.info(f'Lustre health on {label}:')
    ssh_exec(ssh, eip, 'sudo lctl dl', logger=logger)

# --- Client verification ---
if client_nodes:
    first_ssh = client_nodes[0]['ssh']
    first_eip = client_nodes[0]['eip']
    ssh_exec(
        first_ssh,
        first_eip,
        f"echo 'test' | sudo tee /mnt/{FSNAME}/test.txt",
        logger=logger,
    )
    ssh_exec(
        first_ssh,
        first_eip,
        f'cat /mnt/{FSNAME}/test.txt',
        logger=logger,
    )
    for mdt_idx in range(NUM_MDS):
        ssh_exec(
            first_ssh,
            first_eip,
            f'sudo lfs changelog {FSNAME}-MDT{mdt_idx:04d}',
            logger=logger,
        )
    ssh_exec(first_ssh, first_eip, 'lfs df -h', logger=logger)
    ssh_exec(first_ssh, first_eip, 'lfs osts', logger=logger)

logger.info('Step 3 completed: mount, changelog, verification')

## Deployment Summary

Print timing, topology, per-node details, and SSH commands. Save final cluster state.

In [ ]:
# Deployment Summary
end_time = datetime.now()
duration = end_time - start_time

logger.info('=' * 60)
logger.info('DEPLOYMENT COMPLETED SUCCESSFULLY')
logger.info('=' * 60)

total_osts = NUM_OSS * NUM_OST_PER_OSS
logger.info(f'Total deployment time: {duration}')
logger.info(
    f'Topology: 1 MGS + {NUM_MDS} MDS + {NUM_OSS} OSS'
    f' ({total_osts} OSTs) + {NUM_CLIENTS} clients',
)
logger.info(f'AMIs: RHEL={AMI_ID}, Ubuntu={UBUNTU_AMI_ID}')

logger.info('\nDeployed Instances:')
logger.info(f'  {MGS_MDS_INSTANCE_NAME} (MGS+MDS0)')
logger.info(
    f'    Instance: {MGS_MDS_INSTANCE_ID}'
    f' | Public: {MGS_MDS_EIP}'
    f' | Private: {MGS_MDS_PRIVATE_IP}',
)

for node in mds_nodes:
    logger.info(f'  {node["name"]} (MDS)')
    logger.info(
        f'    Instance: {node["instance_id"]}'
        f' | Public: {node["eip"]}'
        f' | Private: {node["private_ip"]}',
    )

for node in oss_nodes:
    logger.info(f'  {node["name"]} (OSS)')
    logger.info(
        f'    Instance: {node["instance_id"]}'
        f' | Public: {node["eip"]}'
        f' | Private: {node["private_ip"]}',
    )

for node in client_nodes:
    logger.info(f'  {node["name"]} (Client)')
    logger.info(
        f'    Instance: {node["instance_id"]} | Public: {node["eip"]}',
    )

logger.info(f'\nFilesystem: {FSNAME}')
logger.info(f'Mount point on clients: /mnt/{FSNAME}')

logger.info('\nSSH Access:')
logger.info(f'  ssh {MGS_MDS_INSTANCE_NAME}')
for node in mds_nodes:
    logger.info(f'  ssh {node["name"]}')
for node in oss_nodes:
    logger.info(f'  ssh {node["name"]}')
for node in client_nodes:
    logger.info(f'  ssh {node["name"]}')

logger.info(f'\nLog file: {log_file}')
logger.info('=' * 60)

# Save final state
_state['completed_at'] = datetime.now().isoformat()
_state['status'] = 'deployed'
save_cluster_state(_state, 'lustre-state.json', logger=logger)